# 🤝 Collaborative Filtering (CF) — Hệ Thống Gợi Ý Điểm Rèn Luyện CTU

## Mục tiêu
Notebook này xây dựng **Item-Based Collaborative Filtering** cho hệ thống gợi ý.

**Ý tưởng cốt lõi:** Tìm các sự kiện *tương tự nhau* dựa trên hành vi của **cộng đồng sinh viên**:
> "Những sinh viên đã tham gia sự kiện A cũng thường tham gia sự kiện B → Nếu bạn thích A, gợi ý B"

**Tại sao Item-Based thay vì User-Based?**
- Số sự kiện (~vài trăm/năm) << số sinh viên (~hàng nghìn) → ma trận item-item ổn định hơn
- Dễ pre-compute và cache similarity matrix

## Interaction Matrix
Ma trận sinh viên × sự kiện với các giá trị:
| Signal | Giá trị |
|--------|--------|
| ATTENDED | 2.0 |
| REGISTERED | 1.0 |
| SUBSCRIBED (category/criteria match) | 0.5 |
| ABSENT | -0.5 |
| CANCELLED | 0.0 |
| Không tương tác | NaN |

> **Lưu ý:** CF cần đủ dữ liệu (≥ 1 học kỳ vận hành) để có ý nghĩa.

---
## Cell 1: Import Thư Viện

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.impute import SimpleImputer

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

random.seed(42)
np.random.seed(42)

print('✅ Import thư viện thành công!')

---
## Cell 2: Tạo Fake Dataset

Tái tạo dataset với **quy mô lớn hơn** để ma trận CF có ý nghĩa (30 sinh viên, 20 sự kiện).
CF cần đủ dữ liệu tương tác mới hoạt động tốt.

In [ ]:
# ============================================================
# Tiêu chí & Danh mục
# ============================================================
leaf_criteria = ['C3_1', 'C3_2', 'C3_3', 'C5_1', 'C5_2']

df_criterias = pd.DataFrame([
    {'criteriaId': 'C3_1', 'criteriaName': 'Hoạt động học thuật',   'maxScore': 10},
    {'criteriaId': 'C3_2', 'criteriaName': 'Hoạt động văn thể mỹ', 'maxScore': 5},
    {'criteriaId': 'C3_3', 'criteriaName': 'Hoạt động tình nguyện','maxScore': 5},
    {'criteriaId': 'C5_1', 'criteriaName': 'Công tác Đoàn - Hội',  'maxScore': 5},
    {'criteriaId': 'C5_2', 'criteriaName': 'Công tác Đội - nhóm',  'maxScore': 5},
])

df_categories = pd.DataFrame([
    {'categoryId': 'CAT1', 'categoryName': 'Học thuật - Nghiên cứu'},
    {'categoryId': 'CAT2', 'categoryName': 'Văn hoá - Nghệ thuật'},
    {'categoryId': 'CAT3', 'categoryName': 'Thể dục - Thể thao'},
    {'categoryId': 'CAT4', 'categoryName': 'Tình nguyện - Cộng đồng'},
    {'categoryId': 'CAT5', 'categoryName': 'Đoàn - Hội sinh viên'},
    {'categoryId': 'CAT6', 'categoryName': 'Kỹ năng - Hướng nghiệp'},
])

# ============================================================
# Sự kiện (tất cả các sự kiện đã diễn ra trong quá khứ để CF học)
# ============================================================
event_templates = [
    ('Hội thảo AI & Machine Learning', 'C3_1', 5,  ['CAT1', 'CAT6']),
    ('Cuộc thi Lập trình Sinh viên', 'C3_1', 8,    ['CAT1']),
    ('Seminar Khoa học Dữ liệu', 'C3_1', 4,        ['CAT1', 'CAT6']),
    ('Workshop Kỹ năng Mềm', 'C3_1', 3,            ['CAT6']),
    ('Ngày hội Học thuật Khoa CNTT', 'C3_1', 5,    ['CAT1']),
    ('Liên hoan Văn nghệ SV', 'C3_2', 5,           ['CAT2']),
    ('Giải Cầu Lông Sinh viên', 'C3_2', 4,         ['CAT3']),
    ('CTU Got Talent', 'C3_2', 6,                  ['CAT2']),
    ('Giải Bóng Đá Khoa', 'C3_2', 5,               ['CAT3']),
    ('Triển lãm Mỹ thuật SV', 'C3_2', 3,           ['CAT2']),
    ('Hiến máu nhân đạo', 'C3_3', 10,              ['CAT4']),
    ('Mùa hè xanh', 'C3_3', 8,                    ['CAT4']),
    ('Nhặt rác bảo vệ môi trường', 'C3_3', 5,     ['CAT4']),
    ('Hỗ trợ đồng bào lũ lụt', 'C3_3', 10,       ['CAT4']),
    ('Trao học bổng trẻ em nghèo', 'C3_3', 7,    ['CAT4']),
    ('Đại hội Đoàn Khoa', 'C5_1', 6,              ['CAT5']),
    ('Kết nạp Đoàn viên mới', 'C5_1', 5,          ['CAT5']),
    ('Hội nghị Ban Chấp hành', 'C5_1', 4,         ['CAT5']),
    ('Chào tân sinh viên', 'C5_2', 5,             ['CAT5', 'CAT4']),
    ('Ngày hội việc làm CTU', 'C5_2', 4,          ['CAT6', 'CAT5']),
]

now = datetime.now()
events_data = []
for i, (name, cid, score, cats) in enumerate(event_templates):
    events_data.append({
        'eventId': f'EVT_{i+1:03d}',
        'eventName': name,
        'criteriaId': cid,
        'score': score,
        'categories': cats,
        'startDate': now - timedelta(days=random.randint(10, 180)),  # Tất cả đã qua
        'is_upcoming': False,
    })

# Thêm sự kiện sắp diễn ra (để gợi ý)
future_events = [
    ('Workshop Deep Learning', 'C3_1', 5, ['CAT1', 'CAT6']),
    ('Gala Văn nghệ Cuối năm', 'C3_2', 6, ['CAT2']),
    ('Hiến máu Tình nguyện T4/2026', 'C3_3', 10, ['CAT4']),
    ('Đại hội SV Khoa CNTT 2026', 'C5_1', 6, ['CAT5']),
]
for i, (name, cid, score, cats) in enumerate(future_events):
    events_data.append({
        'eventId': f'EVT_F_{i+1:02d}',
        'eventName': name,
        'criteriaId': cid,
        'score': score,
        'categories': cats,
        'startDate': now + timedelta(days=random.randint(5, 30)),
        'is_upcoming': True,
    })

df_events = pd.DataFrame(events_data)
past_events = df_events[~df_events['is_upcoming']]['eventId'].tolist()
upcoming_events = df_events[df_events['is_upcoming']]['eventId'].tolist()

# ============================================================
# Sinh viên (30 người để CF có đủ dữ liệu)
# ============================================================
student_ids = [f'SV_{i:03d}' for i in range(1, 31)]
df_students = pd.DataFrame({'studentId': student_ids})

# ============================================================
# Subscription (explicit preference)
# ============================================================
subs_data = []
all_cat_ids = df_categories['categoryId'].tolist()
for sid in student_ids:
    if random.random() < 0.7:
        subs_data.append({
            'studentId': sid,
            'subscribed_categories': random.sample(all_cat_ids, k=random.randint(1, 3)),
            'subscribed_criteria': random.sample(leaf_criteria, k=random.randint(1, 2)),
        })
df_subscriptions = pd.DataFrame(subs_data)

# ============================================================
# Lịch sử đăng ký (event_registrations)
# Mô phỏng mỗi sinh viên tham gia 4-10 sự kiện đã qua
# ============================================================
reg_data = []
for sid in student_ids:
    joined = random.sample(past_events, k=random.randint(4, 10))
    for eid in joined:
        status = random.choices(
            ['ATTENDED', 'REGISTERED', 'ABSENT', 'CANCELLED'],
            weights=[0.65, 0.15, 0.12, 0.08]
        )[0]
        reg_data.append({'studentId': sid, 'eventId': eid, 'status': status})

df_registrations = pd.DataFrame(reg_data).drop_duplicates(['studentId', 'eventId'])

print(f'✅ Dataset đã tạo:')
print(f'   Sinh viên     : {len(df_students)}')
print(f'   Sự kiện (quá khứ): {len(past_events)}')
print(f'   Sự kiện (tương lai): {len(upcoming_events)}')
print(f'   Lượt đăng ký  : {len(df_registrations)}')
print(f'   Có subscription: {len(df_subscriptions)}')

---
## Cell 3: Xây Dựng Interaction Matrix

Tạo ma trận **sinh viên × sự kiện** với các giá trị tương tác.

Đây là bước cốt lõi của CF. Ma trận thưa (sparse) là bình thường — mỗi sinh viên chỉ tham gia một phần nhỏ sự kiện.

**Lưu ý quan trọng:** Subscription được đưa vào ma trận như một tương tác ngầm định (giá trị 0.5) cho tất cả sự kiện thuộc category/criteria đã subscribe.

In [ ]:
# Ánh xạ status → giá trị tương tác
STATUS_WEIGHTS = {
    'ATTENDED'   : 2.0,
    'REGISTERED' : 1.0,
    'ABSENT'     : -0.5,
    'CANCELLED'  : 0.0,   # Bỏ qua (không biết lý do)
}
SUBSCRIPTION_WEIGHT = 0.5  # Explicit preference nhưng chưa phải quyết định cụ thể

# Chỉ dùng sự kiện đã diễn ra để xây interaction matrix
df_past_events = df_events[~df_events['is_upcoming']].copy()

# ============================================================
# Bước 1: Khởi tạo ma trận NaN (chưa có tương tác)
# ============================================================
interaction_matrix = pd.DataFrame(
    np.nan,
    index=student_ids,
    columns=past_events
)

# ============================================================
# Bước 2: Điền từ event_registrations
# ============================================================
for _, row in df_registrations.iterrows():
    sid, eid, status = row['studentId'], row['eventId'], row['status']
    if eid in interaction_matrix.columns:
        weight = STATUS_WEIGHTS.get(status, 0.0)
        if not np.isnan(weight):
            interaction_matrix.loc[sid, eid] = weight

# ============================================================
# Bước 3: Điền từ Subscription (implicit interest)
# Nếu ô còn NaN và sv đã subscribe category/criteria của sự kiện đó → 0.5
# ============================================================
for _, sub_row in df_subscriptions.iterrows():
    sid = sub_row['studentId']
    sub_cats = sub_row['subscribed_categories']
    sub_crits = sub_row['subscribed_criteria']

    for _, evt in df_past_events.iterrows():
        eid = evt['eventId']
        # Chỉ điền nếu chưa có tương tác trực tiếp
        if np.isnan(interaction_matrix.loc[sid, eid]):
            cat_match = any(c in sub_cats for c in evt['categories'])
            crit_match = evt['criteriaId'] in sub_crits
            if cat_match or crit_match:
                interaction_matrix.loc[sid, eid] = SUBSCRIPTION_WEIGHT

# Thống kê
total_cells = interaction_matrix.shape[0] * interaction_matrix.shape[1]
filled_cells = interaction_matrix.notna().sum().sum()
sparsity = 1 - filled_cells / total_cells

print(f'📊 Interaction Matrix: {interaction_matrix.shape[0]} sinh viên × {interaction_matrix.shape[1]} sự kiện')
print(f'   Tổng ô: {total_cells}')
print(f'   Ô có dữ liệu: {filled_cells} ({filled_cells/total_cells*100:.1f}%)')
print(f'   Sparsity: {sparsity:.1%} (thưa = bình thường với hệ thống CF)')
print()
print('Mẫu ma trận (5 sinh viên × 5 sự kiện đầu):')
print(interaction_matrix.iloc[:5, :5].to_string())

---
## Cell 4: Tính Item-Item Similarity

Tính **Cosine Similarity** giữa các cặp sự kiện dựa trên ma trận tương tác.

Sự kiện A và B tương tự nhau nếu **cùng một nhóm sinh viên** thích cả hai.

**Xử lý NaN:** Điền bằng 0 trước khi tính cosine (sinh viên chưa tương tác = 0).

In [ ]:
# Điền NaN = 0 (không tương tác)
matrix_filled = interaction_matrix.fillna(0).values  # Shape: (n_students, n_events)

# Tính item-item similarity: transpose để các cột (events) trở thành hàng
item_features = matrix_filled.T  # Shape: (n_events, n_students)
item_similarity = cosine_similarity(item_features)  # Shape: (n_events, n_events)

# Đóng gói thành DataFrame để dễ đọc
event_ids_past = interaction_matrix.columns.tolist()
df_item_sim = pd.DataFrame(
    item_similarity,
    index=event_ids_past,
    columns=event_ids_past
)

print(f'📐 Item-Item Similarity Matrix: {df_item_sim.shape}')
print()

# Kiểm tra: tìm 3 sự kiện tương tự nhất với EVT_001 (Hội thảo AI)
target_event = 'EVT_001'
target_name = df_events[df_events['eventId']==target_event]['eventName'].values[0]
print(f'🔍 3 sự kiện tương tự nhất với "{target_name}" ({target_event}):')
similar = df_item_sim[target_event].drop(target_event).sort_values(ascending=False).head(3)
for eid, sim_score in similar.items():
    ename = df_events[df_events['eventId']==eid]['eventName'].values[0]
    cid = df_events[df_events['eventId']==eid]['criteriaId'].values[0]
    print(f'   [{sim_score:.4f}] {eid}: {ename} ({cid})')

print()
print('Ma trận similarity (5×5 đầu):')
print(df_item_sim.iloc[:5, :5].to_string())

---
## Cell 5: Tính Predicted Score Cho Sự Kiện Chưa Tham Gia

Với một sinh viên `s` và một sự kiện chưa tham gia `e`, dự đoán điểm của `s` với `e` dựa trên:
- Các sự kiện `e'` mà `s` đã tương tác
- Mức độ tương đồng giữa `e` và `e'`

**Công thức:**
```
pred(s, e) = Σ [ sim(e, e') × rating(s, e') ] / Σ |sim(e, e')|
           (với e' là sự kiện sinh viên s đã tương tác)
```

In [ ]:
def predict_cf_score(student_id: str, target_event_id: str, top_k_similar: int = 5) -> float:
    """
    Dự đoán điểm tương tác của sinh viên với một sự kiện cụ thể.

    Args:
        student_id      : ID sinh viên
        target_event_id : Event ID muốn dự đoán
        top_k_similar   : Số lượng sự kiện tương tự nhất để tính weighted average

    Returns:
        Predicted score (float). NaN nếu không thể dự đoán.
    """
    if target_event_id not in df_item_sim.columns:
        return np.nan  # Sự kiện mới chưa có trong interaction matrix

    # Lấy tất cả sự kiện mà sinh viên đã tương tác (không NaN)
    student_ratings = interaction_matrix.loc[student_id].dropna()

    if len(student_ratings) == 0:
        return np.nan  # Cold-start: không có dữ liệu

    # Lấy similarity của target event với các sự kiện sinh viên đã tương tác
    sim_scores = df_item_sim.loc[target_event_id, student_ratings.index]

    # Lấy top-k sự kiện tương tự nhất
    top_k = sim_scores.nlargest(top_k_similar)

    numerator = (top_k * student_ratings[top_k.index]).sum()
    denominator = top_k.abs().sum()

    if denominator == 0:
        return 0.0

    return numerator / denominator


# Kiểm tra
test_sv = 'SV_001'
test_evt = 'EVT_002'
pred = predict_cf_score(test_sv, test_evt)
actual = interaction_matrix.loc[test_sv, test_evt]
evt_name = df_events[df_events['eventId']==test_evt]['eventName'].values[0]

print(f'🧪 Kiểm tra predict cho {test_sv} với sự kiện "{evt_name}":')
print(f'   Predicted score: {pred:.4f}')
print(f'   Actual value   : {actual} ({"có tương tác" if not np.isnan(actual) else "chưa tương tác"})')
print()
print('✅ Hàm predict_cf_score() hoạt động!')

---
## Cell 6: Hàm Gợi Ý CF

Gợi ý sự kiện **sắp diễn ra** cho sinh viên:
1. Lọc sự kiện sinh viên chưa đăng ký
2. Với mỗi sự kiện sắp diễn ra, tìm sự kiện tương tự đã qua trong interaction matrix
3. Tính predicted score và xếp hạng

In [ ]:
def recommend_cf(student_id: str, top_n: int = 5) -> pd.DataFrame:
    """
    Gợi ý sự kiện sắp diễn ra bằng Item-Based CF.

    Với sự kiện chưa xuất hiện trong interaction matrix (sự kiện mới),
    CF sẽ tìm sự kiện tương tự (dựa trên feature: criteriaId, categories)
    trong ma trận để ước lượng.
    """
    registered = set(df_registrations[
        df_registrations['studentId'] == student_id
    ]['eventId'].tolist())

    results = []
    for _, evt in df_events[df_events['is_upcoming']].iterrows():
        eid = evt['eventId']

        # Bỏ qua sự kiện đã đăng ký
        if eid in registered:
            continue

        # Sự kiện sắp diễn ra chưa nằm trong interaction matrix
        # → Tìm sự kiện tương tự trong past events dựa trên criteria + category
        # → Dùng trung bình predicted scores của các sự kiện tương tự
        similar_past = df_events[
            (~df_events['is_upcoming']) &
            (df_events['criteriaId'] == evt['criteriaId'])
        ]['eventId'].tolist()

        if not similar_past:
            cf_score = 0.0
        else:
            # Tính average predicted score qua các sự kiện tương tự
            preds = [predict_cf_score(student_id, past_eid) for past_eid in similar_past]
            preds = [p for p in preds if not np.isnan(p)]
            cf_score = np.mean(preds) if preds else 0.0

        results.append({
            'eventId'   : eid,
            'eventName' : evt['eventName'],
            'criteriaId': evt['criteriaId'],
            'score'     : evt['score'],
            'cf_score'  : round(cf_score, 4),
        })

    return pd.DataFrame(results).sort_values('cf_score', ascending=False).head(top_n).reset_index(drop=True)


print('✅ Hàm recommend_cf() đã sẵn sàng!')

---
## Cell 7: Test Gợi Ý CF Cho Nhiều Sinh Viên

In [ ]:
test_students = ['SV_001', 'SV_002', 'SV_005', 'SV_010', 'SV_015']

for sid in test_students:
    # Thống kê lịch sử của sinh viên
    history = df_registrations[df_registrations['studentId'] == sid]
    attended = history[history['status'] == 'ATTENDED']['eventId'].tolist()

    # Lấy gợi ý
    recs = recommend_cf(sid, top_n=3)

    print(f'┌─ {sid} (lịch sử: {len(history)} sự kiện, tham dự thật: {len(attended)}) ───────────')
    if recs.empty:
        print('│  ⚠️  Không gợi ý được (cold-start)')
    else:
        for _, row in recs.iterrows():
            print(f'│  [{row["cf_score"]:.4f}] {row["eventName"]} ({row["criteriaId"]}, {row["score"]} điểm)')
    print()

---
## Cell 8: Phân Tích Sparsity & Cold-Start Problem

Minh họa vấn đề quan trọng của CF: **ma trận càng thưa → gợi ý càng kém chính xác**.

In [ ]:
print('📊 PHÂN TÍCH VẤN ĐỀ SPARSITY & COLD-START')
print('='*55)
print()

# Tính số tương tác và CF score trung bình cho từng sinh viên
student_stats = []
for sid in student_ids[:20]:
    history = df_registrations[df_registrations['studentId'] == sid]
    n_interactions = interaction_matrix.loc[sid].notna().sum()
    recs = recommend_cf(sid, top_n=4)
    avg_score = recs['cf_score'].mean() if not recs.empty else 0
    student_stats.append({
        'studentId': sid,
        'n_registrations': len(history),
        'n_interactions_in_matrix': n_interactions,
        'avg_cf_rec_score': round(avg_score, 4),
    })

df_stats = pd.DataFrame(student_stats)
print(df_stats.to_string(index=False))

print()
print('💡 Nhận xét:')
print('  - Sinh viên có nhiều tương tác → CF score gợi ý cao hơn')
print('  - Sinh viên ít tương tác → CF score thấp/0 (cold-start)')
print('  - Giải pháp: kết hợp với CBF qua Hybrid Filtering (xem notebook 03)')

---
## Cell 9: Tổng Kết CF

**Ưu điểm:**
- ✅ Học từ hành vi cộng đồng → đa dạng hơn CBF
- ✅ Không cần item features (chỉ cần tương tác)
- ✅ Tự động phát hiện pattern ẩn (sv thích học thuật thường cũng thích kỹ năng)

**Nhược điểm:**
- ❌ Cold-start: sinh viên mới / sự kiện mới → không gợi ý được
- ❌ Cần tích lũy đủ dữ liệu (≥ 1 học kỳ)
- ❌ Ma trận thưa → accuracy thấp hơn

**Bước tiếp theo:** Kết hợp CBF + CF trong notebook `03_hybrid_filtering.ipynb`

In [ ]:
print('📊 Tổng kết Item-Based Collaborative Filtering')
print('='*55)
print(f'Kích thước interaction matrix: {interaction_matrix.shape}')
print(f'Sparsity: {sparsity:.1%}')
print()
print('Phân phối giá trị trong matrix:')
flat_vals = interaction_matrix.values.flatten()
flat_vals = flat_vals[~np.isnan(flat_vals)]
for val, label in [(2.0, 'ATTENDED'), (1.0, 'REGISTERED'), (0.5, 'SUBSCRIBED'), (-0.5, 'ABSENT'), (0.0, 'CANCELLED')]:
    count = (flat_vals == val).sum()
    print(f'  {label:15s} ({val:+.1f}): {count:3d} ô  {"█" * (count // 2)}')
print()
print('Item-Item Similarity thống kê:')
sim_vals = item_similarity.flatten()
sim_vals = sim_vals[sim_vals < 1.0]  # Bỏ diagonal
print(f'  Min  : {sim_vals.min():.4f}')
print(f'  Max  : {sim_vals.max():.4f}')
print(f'  Mean : {sim_vals.mean():.4f}')

---
## Cell 10: Chuẩn Hóa CF Score (Min-Max) Cho Hybrid

Khi kết hợp với CBF trong Hybrid, nên chuẩn hóa CF score trước khi trộn:
```
x_norm = (x - min) / (max - min)
```

Giữ cả `cf_score` gốc để giải thích và debug.

In [ ]:
def minmax_normalize(scores: dict) -> dict:
    """Min-Max normalization: x_norm = (x - min) / (max - min)."""
    vals = [v for v in scores.values() if v is not None and not np.isnan(v)]
    if not vals or max(vals) == min(vals):
        return {k: (np.nan if (v is None or np.isnan(v)) else 0.5) for k, v in scores.items()}
    lo, hi = min(vals), max(vals)
    return {k: (np.nan if (v is None or np.isnan(v)) else (v - lo) / (hi - lo)) for k, v in scores.items()}


def recommend_cf_with_norm(student_id: str, top_n: int = 5) -> pd.DataFrame:
    """Trả về cả cf_score gốc và cf_norm để dùng cho Hybrid."""
    raw_df = recommend_cf(student_id=student_id, top_n=1000)
    raw_scores = {row['eventId']: float(row['cf_score']) for _, row in raw_df.iterrows()}
    norm_scores = minmax_normalize(raw_scores)
    raw_df['cf_norm'] = raw_df['eventId'].map(norm_scores).astype(float)
    return raw_df.sort_values('cf_norm', ascending=False).head(top_n).reset_index(drop=True)


print('✅ Đã thêm chuẩn hóa CF theo Min-Max cho Hybrid')
print('   Công thức: x_norm = (x - min) / (max - min)')
print()
demo_cf_norm = recommend_cf_with_norm('SV_001', top_n=5)
print(demo_cf_norm[['eventName', 'cf_score', 'cf_norm']].to_string(index=False))